# Повтор только Kev на закреплённых новых входах

Пять Nano/Laya расчётов уже завершены в версии 2 и сохранены локально. Повторяем original Kev и выбранный шаг710 после исправления имени базового shard в метаданных отчёта. Входы, веса и FP32 inference неизменны; optimizer не создаётся. Подключены только два нужных датасета и завершённый training notebook Kev.

In [ ]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'laya==0.3.26', 'peft==0.19.0', 'transformers==5.6.2', 'safetensors==0.6.2', 'accelerate==1.15.0', 'pydantic==2.12.5'])


In [ ]:
matches = list(Path('/kaggle/input').rglob('fixed-checkpoints.json'))
assert len(matches) == 1, 'Evaluation dataset is missing or ambiguous'
input_dir = matches[0].parent
assert hashlib.sha256((input_dir / 'input-manifest.json').read_bytes()).hexdigest() == '2c48d9f74a6916b8c015a9cbe52a69e758282542f56c16cd8878573910d4fa7b', 'Unexpected evaluation version'
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir / 'evaluate_decision_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir)]
command += ['--models', 'original-kev', 'trained-kev']
with Path('/kaggle/working/evaluation.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'Inspect evaluation.log'


## Проверка результата

Все строки остаются в знаменателях. ASR — шумная TTS-речь после настоящего STT; человеческих голосов здесь нет. Вероятности не откалиброваны. GPU inference не измеряет скорость локального CPU. После скачивания проводится аудит SHA-256, ID, вероятностей, метрик и парных различий.

In [ ]:
completion = json.loads((output_dir / 'completion.json').read_text())
assert len(completion['completed_models']) == 2 and completion['optimizer_updates'] == 0
print(completion)
